# Cobblemon Atlas｜烈空座安全鏡頭 V2

48 幀取樣、每幀安全邊界檢查、裁切自動回退固定鏡頭；只輸出測試候選，不覆蓋圖鑑。

In [ ]:
from pathlib import Path
import sys,subprocess,zipfile,shutil,json
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK=Path('/content/cobblemon-bedrock')
SITE=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
assert ROOT.is_dir(),'找不到 Cobblemon_Atlas 資料夾'

In [ ]:
if not (SITE/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE)],check=True)
subprocess.run(['git','-C',str(SITE),'sparse-checkout','set','tools','docs','notebooks'],check=True)
subprocess.run(['git','-C',str(SITE),'pull','--ff-only','origin',BRANCH],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(SITE),'rev-parse','--short','HEAD'],text=True))

In [ ]:
WORK.mkdir(parents=True,exist_ok=True)
for name in ['merged_pack','development_resource_packs']:
    target=WORK/name
    if not target.is_dir():
        zpath=ROOT/'source_packs'/(name+'.zip')
        assert zpath.is_file(),zpath
        with zipfile.ZipFile(zpath) as z:
            for entry in z.infolist():
                dst=(WORK/entry.filename).resolve()
                assert dst.is_relative_to(WORK.resolve()),'Unsafe archive path'
            z.extractall(WORK)
    assert target.is_dir(),target
(WORK/'tools').mkdir(exist_ok=True)
shutil.copy2(ROOT/'source_packs/render_portraits.py',WORK/'tools/render_portraits.py')
print('原始素材已備妥')

In [ ]:
report=ROOT/'reports/resolver-validation/resolved-forms.json'
assert report.is_file(),report
out=ROOT/'reports/resolver-validation/safe-camera-pilot'
cmd=[sys.executable,str(SITE/'tools/render_safe_camera_pilot.py'),'--repo',str(WORK),'--resolved',str(report),'--out',str(out)]
subprocess.run(cmd,check=True)
result=json.loads((out/'safe-camera-report.json').read_text())
for k,v in result['trials'].items(): print(k,v)
print('REPORT:',out/'safe-camera-report.json')